# 🚀 Universal Multi-Company Stock LSTM (Google Colab)
### Train ONE Unified Deep Learning Model Across All Company Datasets

This notebook enables you to:
1. **Upload your own company CSV files or a ZIP folder directly from your device** (or use the built-in 55 NIFTY 50 dataset).
2. **Normalize each stock independently** to prevent expensive stocks (e.g. ₹3,500) from dominating cheaper stocks (e.g. ₹100).
3. **Train ONE universal PyTorch LSTM** with **AdamW** and **ReduceLROnPlateau** on GPU.
4. **Click or select any company from an interactive dropdown** to immediately see its multi-step forecast cone, test accuracy, and directional trend.

---
## 1. Hardware Check & Environment Setup
Make sure GPU acceleration is enabled: Go to **Runtime** → **Change runtime type** → select **T4 GPU**.

In [ ]:
# Verify GPU hardware
!nvidia-smi

# Install required dependencies
!pip install -q torch pandas numpy matplotlib scikit-learn yfinance ipywidgets

---
## 2. Clone GitHub Repository
Clones the repository containing the model architecture, feature engineering pipeline, and built-in datasets.

In [ ]:
import os

REPO_URL = "https://github.com/Raghava-sensai/stock-lstm-prediction.git"
REPO_DIR = "stock-lstm-prediction"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

print(f"\n[OK] Current Active Directory: {os.getcwd()}")

---
## 3. Data Ingestion: Upload from Device OR Use Pre-Loaded Data
You can choose between:
- **Option A (Device Upload)**: Upload your own CSV files or a ZIP archive containing CSVs from your computer.
- **Option B (Pre-loaded GitHub Data)**: Use the 55 NIFTY 50 company CSV files included in the repository.

In [ ]:
#@title 📂 Select Dataset Source { run: "auto" }
DATA_SOURCE = "Upload from Device (CSV or ZIP)" #@param ["Upload from Device (CSV or ZIP)", "Use Pre-loaded NIFTY 50 (55 Stocks)"]

import os
import glob
import shutil
import zipfile
import pandas as pd

UPLOAD_DIR = "uploaded_company_data"
TARGET_DATA_DIR = ""

if DATA_SOURCE == "Upload from Device (CSV or ZIP)":
    from google.colab import files
    os.makedirs(UPLOAD_DIR, exist_ok=True)
    
    print("\n" + "="*65)
    print("[*] Please click 'Choose Files' below to upload CSV files or a ZIP:")
    print("    - You can select MULTIPLE .csv files at once")
    print("    - OR upload a single .zip file containing company CSVs")
    print("="*65 + "\n")
    
    uploaded = files.upload()
    
    for fname, content in uploaded.items():
        dest_path = os.path.join(UPLOAD_DIR, fname)
        with open(dest_path, "wb") as f:
            f.write(content)
        
        # Auto-extract if user uploaded a zip
        if fname.lower().endswith(".zip"):
            print(f"[+] Extracting ZIP archive: {fname}...")
            with zipfile.ZipFile(dest_path, "r") as z:
                z.extractall(UPLOAD_DIR)
            print(f"[OK] Extracted contents into '{UPLOAD_DIR}/'")
        else:
            print(f"[+] Saved CSV: {fname}")
            
    TARGET_DATA_DIR = UPLOAD_DIR
else:
    TARGET_DATA_DIR = "dataset/archive/NifSent/NIFTY 50"

# Scan and report all discovered CSV files
found_csvs = set(glob.glob(os.path.join(TARGET_DATA_DIR, "*.csv")))
found_csvs.update(glob.glob(os.path.join(TARGET_DATA_DIR, "**", "*.csv"), recursive=True))
csv_list = sorted(list(found_csvs))

print(f"\n" + "="*65)
print(f"[SUMMARY] Ready to train on {len(csv_list)} company CSV datasets in '{TARGET_DATA_DIR}':")
print("="*65)
company_names = [os.path.splitext(os.path.basename(f))[0].upper() for f in csv_list]
for i in range(0, len(company_names), 6):
    print("  ".join(f"{c:<12}" for c in company_names[i:i+6]))
print("="*65)

---
## 4. Train ONE Universal Model on All Companies (GPU)
This trains a single PyTorch Stacked LSTM with **AdamW** across all company files.
Each stock is scaled independently to prevent price scale bias, and sliding sequences are pooled into a master training set.

In [ ]:
#@title ⚙️ Training Hyperparameters { run: "auto" }
epochs = 20 #@param {type:"integer"}
batch_size = 256 #@param [64, 128, 256, 512] {type:"raw"}
forecast_horizon = 5 #@param {type:"integer"}
seq_length = 24 #@param {type:"integer"}

output_directory = "./outputs/universal_multi_company"

# Run Universal Multi-Company Training on GPU
!python multi_company_unified_lstm.py \
    --data_dir "{TARGET_DATA_DIR}" \
    --epochs {epochs} \
    --batch_size {batch_size} \
    --seq_length {seq_length} \
    --forecast_horizon {forecast_horizon} \
    --output_dir "{output_directory}"

---
## 5. Multi-Company Leaderboard
Inspect model performance across all companies sorted by lowest test MAPE (Mean Absolute Percentage Error).

In [ ]:
import pandas as pd
from IPython.display import display

leaderboard_path = "outputs/universal_multi_company/multi_company_leaderboard.csv"
if os.path.exists(leaderboard_path):
    df_board = pd.read_csv(leaderboard_path)
    print(f"\n[+] Master Multi-Company Leaderboard ({len(df_board)} Companies):")
    display(df_board.head(30))
else:
    print("[!] Run training in Section 4 first to generate leaderboard.")

---
## 6. 🎯 Interactive Stock Selector & Forecast Viewer
Pick **any company** from the dropdown below to immediately view:
1. **Model Prediction vs Ground Truth** on historical test data.
2. **Real-time 5-Candle Projected Trajectory Cone** with Bullish/Bearish signal and price targets.

In [ ]:
import pickle
from multi_company_unified_lstm import view_company_forecast, launch_interactive_widget

cache_file = "outputs/universal_multi_company/eval_results.pkl"
with open(cache_file, "rb") as f:
    eval_results = pickle.load(f)

# Launch interactive dropdown widget (dynamically includes all uploaded companies!)
launch_interactive_widget(eval_results)

---
### Direct Single-Stock Lookup (Colab Form Field)
You can also type any company ticker name directly below and run:

In [ ]:
#@title 📈 Quick Ticker Lookup { run: "auto" }
target_ticker = "RELIANCE" #@param {type:"string"}

clean_ticker = target_ticker.strip().upper()
if clean_ticker in eval_results:
    view_company_forecast(clean_ticker, eval_results, show_plot=True)
else:
    available = sorted(list(eval_results.keys()))
    print(f"[!] Ticker '{clean_ticker}' not found. Available tickers:\n  {available}")